In [ ]:
import sys
import os
from pathlib import Path

if 'google.colab' in sys.modules:
    print("Ambiente Colab rilevato. Inizializzazione in corso...")
    
    # 1. Monta Drive (se non già montato)
    from google.colab import drive
    if not os.path.exists('/content/drive'):
        drive.mount('/content/drive')
        
    # Imposta la root del Drive su Colab
    BASE_DIR = Path('/content/drive/MyDrive/Progetto_MLDM')
    
    # 2. Clona/Aggiorna la repo
    REPO_DIR = '/content/crop-spatial-classification'
    if not os.path.exists(REPO_DIR):
        !git clone https://github.com/SimoRinaldi/crop-spatial-classification.git
    else:
        !cd {REPO_DIR} && git pull
    
    # 3. Aggancia la cartella per permettere gli import
    if REPO_DIR not in sys.path:
        sys.path.append(REPO_DIR)
        
    # 4. Installa i requisiti in modo silenzioso
    %pip install -q -r {REPO_DIR}/requirements.txt
    print("Setup ambiente Colab completato!")
    
else:
    print("Ambiente Locale rilevato. Procedo con l'esecuzione...")
    
    BASE_DIR = Path.cwd().parent

    if str(BASE_DIR) not in sys.path:
        sys.path.append(str(BASE_DIR))

# =============================
# DEFINIZIONE PATH UNIVERSALI
# =============================
DATA_DIR = BASE_DIR / 'data'
RAW_DIR = DATA_DIR / 'raw'
INTERIM_DIR = DATA_DIR / 'interim'
PROCESSED_DIR = DATA_DIR / 'processed'

# Test di verifica per assicurarsi che i dati siano accessibili
if RAW_DIR.exists():
    print(f"✅ Collegamento ai dati riuscito! Cartella raw: {RAW_DIR}")
else:
    print(f"❌ Attenzione: Cartella non trovata in {RAW_DIR}. Verifica il nome, il mount o il path locale.")

In [ ]:
import os
import glob
import pandas as pd
import rasterio
import rioxarray
import numpy as np
import concurrent.futures
from pathlib import Path
from tqdm.auto import tqdm

path_ground_truth = DATA_DIR / "interim" / "points.json"
path_sentinel2_data = DATA_DIR / "processed" / "sentinel2_data"

df_ground_truth = pd.read_json(path_ground_truth)

def process_single_point(args):
    index, row_dict, base_data_dir = args
    crop_id = row_dict["code"]

    # cerca il file .tif scaricato dentro la cartella del punto (es. point_0/)
    point_dir = base_data_dir / f"point_{index}"
    tif_files = list(point_dir.glob("sentinel2_data_*.tif"))

    if not tif_files:
        return index, None

    point_tif = tif_files[0]

    try:
        with rasterio.open(point_tif) as src:
            data = src.read() # shape: (n_layers, height, width)
            n_layers = src.count

            if n_layers < 6:
                return index, None

            # calcola la media ignorando NaN e pixel nulli (0 = nodata)
            def extract_clean_band_mean(band_index):
                layer_means = []
                for l in range(band_index, n_layers, 6):
                    layer_data = data[l]
                    valid_pixels = layer_data[layer_data > 0]
                    if len(valid_pixels) > 0:
                        layer_means.append(np.nanmean(valid_pixels))
                return float(np.nanmean(layer_means)) if layer_means else 0.0

            # Estraggo i valori medi per le 6 bande spettrometriche:
            # 0: B02 (Blu), 1: B03 (Verde), 2: B04 (Rosso), 3: B08 (NIR), 4: B11 (SWIR1), 5: B12 (SWIR2)
            blue_mean = extract_clean_band_mean(0)                                                                                                                 
            green_mean = extract_clean_band_mean(1)                                                                                                               
            red_mean = extract_clean_band_mean(2)                                                                                                               
            nir_mean = extract_clean_band_mean(3)                                                                                                                 
            swir1_mean = extract_clean_band_mean(4)                                                                                                               
            swir2_mean = extract_clean_band_mean(5)    

    except Exception:
        return index, None

    # calcolo indici spettrali
    den_ndvi = (nir_mean + red_mean)
    ndvi = (nir_mean - red_mean) / den_ndvi if den_ndvi > 0 else 0.0

    den_ndwi = (nir_mean + swir1_mean)
    ndwi = (nir_mean - swir1_mean) / den_ndwi if den_ndwi > 0 else 0.0

    if blue_mean > 0:
        return index, {
            "ID_Campo": index,
            "Ground_Truth": crop_id,
            "Blu_B02": blue_mean,
            "Verde_B03": green_mean,
            "Rosso_B04": red_mean,
            "NIR_B08": nir_mean,
            "SWIR1_B11": swir1_mean,
            "SWIR2_B12": swir2_mean,
            "NDVI": ndvi,
            "NDWI": ndwi,
        }
    return index, None

# preparazione task in parallelo
tasks = [(index, row.to_dict(), path_sentinel2_data) for index, row in df_ground_truth.iterrows()]
results = [None] * len(tasks)
workers = min(16, os.cpu_count() or 4)

print(f"Avvio estrazione feature su {len(tasks)} campi con {workers} worker...")

with concurrent.futures.ProcessPoolExecutor(max_workers=workers) as executor:
    futures = [executor.submit(process_single_point, t) for t in tasks]

    for future in tqdm(concurrent.futures.as_completed(futures), total=len(futures), desc="Elaborazione campi"):
        idx, res = future.result()
        if res is not None:
            results[idx] = res

# crea il dataframe finale 
valid_results = [r for r in results if r is not None]
final_df = pd.DataFrame(valid_results)
print(f"Estrazione completata!Dataset creato con {len(final_df)} campi validi.")


In [ ]:
dataset_dir = DATA_DIR / 'processed' / 'dataset'
dataset_dir.mkdir(parents=True, exist_ok=True)

# salva il dataset in formato Parquet
parquet_path = dataset_dir / 'dataset.parquet'
final_df.to_parquet(parquet_path, index=False)

# salva il dataset in formato CSV
csv_path = dataset_dir / 'dataset.csv'
final_df.to_csv(csv_path, index=False)

print(f"Dataset salvati con successo nella cartella: {dataset_dir.resolve()}")       
print(f"\nDimensioni tabella: {final_df.shape[0]} righe x {final_df.shape[1]} colonne\n")

# mostra le prime 5 righe del dataset
display(final_df.head())